This notebook is for modeling and evaluating span identification from the SemEval dataset. This is the first filter for propaganda, so we only want to filter out what we are confident is not propaganda (so high-sensitivity/high-recall). Then downstream, let the technique classification (TC) model handle the precision and pruning.

Best performance of RoBERTa model (adjusting parameters to optimize F2 score, then adjusted threshold to attempt to reach 0.9 recall) using BIO tagging technique rather than multi-class approach:

Model Performance with LR=2.5e-05, WD=0.15

| Recall | Precision | F1 Score | F2 Score |
| :--- | :--- | :--- | :--- |
| 0.9198 | 0.2235 | 0.3596 | 0.5667 |

To implement the Claimify approach rather than the binary "Propaganda vs. Not" approach, we are treated this as a Categorical Span Identification task. The idea is that not all propaganda is created equal, and different techniques are very different from each other linguistically.

In [ ]:
import os
import json
import torch
from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments,
    Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback)
from sklearn.metrics import precision_recall_fscore_support
from datasets import Dataset
from accelerate.state import AcceleratorState
import zipfile
import shutil
import gdown
import evaluate

In [ ]:
AcceleratorState._reset_state()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_roberta_scanner"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#Download models if not already
def setup_models(file_id, target_path):
    zip_temp = target_path.with_suffix(".zip")

    if not (target_path / "model.safetensors").exists() and not (target_path / "pytorch_model.bin").exists():
        print(f"Model not found. Downloading {target_path} from Google Drive...")
        target_path.parent.mkdir(exist_ok=True, parents=True)

        url = f'https://drive.google.com/uc?id={file_id}'

        try:
            gdown.download(url, str(zip_temp), quiet=False)

            print("Extracting...")
            with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
                internal_zip_folder = zip_ref.namelist()[0].split('/')[0]
                zip_ref.extractall(target_path.parent)

            extracted_path = target_path.parent / internal_name
            if extracted_path != target_path:
                if target_path.exists(): shutil.rmtree(target_path)
                os.rename(extracted_path, target_path)

            os.remove(zip_temp)
            print(f"Setup complete: {target_path}")
            return True
        except Exception as e:
            print(f"Download failed: {e}")
            return False
    else:
        print(f"Model weights detected locally at {target_path}")
        return True


#Identify if model exists
model_exists = setup_models('1yaMabdQaYd6CNgdMUn2-sXSo2cTpUATs', MODEL_DIR)

In [ ]:
#Load article-level span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si['propaganda_offsets'] = df_si['propaganda_offsets'].apply(json.loads)
print(f"Loaded {len(df_si)} articles.")
df_si.head()

In [ ]:
#Initialize the model tokenizer
##Tried "bert-base-uncased", Best F1 score after 3 epochs: 0.26392
##"microsoft/deberta-v3-small", After 3 epoches, still getting gradient explosion every time, even after adjusting hyperparameters
raw_dataset = Dataset.from_pandas(df_si)
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_inputs.pop("offset_mapping")
    labels = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        article_spans = examples["propaganda_offsets"][sample_idx]
        doc_labels = []
        for start, end in offsets:
            if start == end == 0:
                doc_labels.append(-100)
                continue
            is_prop = any(s <= start < e or s < end <= e for s, e in article_spans)
            doc_labels.append(1 if is_prop else 0)
        labels.append(doc_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

#Tokenize and split data
tokenized_datasets = raw_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_dataset.column_names).train_test_split(test_size=0.2, seed=42)

In [ ]:
#Tried without weighting before and was quickly overfitting, so weight now
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        #Prioritize Recall: Propaganda classes (1, 2) weighted 20x more than background (0)
        weights = torch.tensor([1.0, 20.0, 20.0], device = model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(p):
    logits, labels = p
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    prop_probs = probs[:, :, 1] + probs[:, :, 2] # Sum of B and I labels

    y_true = labels.flatten()
    mask = y_true != -100
    y_true_clean = (y_true[mask] > 0).astype(int)
    prop_probs_clean = prop_probs.flatten()[mask]

    #Search from 0.1 to 0.50
    thresholds = np.arange(0.1, 0.51, 0.02)
    best_metrics = {"precision": 0, "recall": 0, "f1": 0, "threshold": 0.01}

    target_recall = 0.90
    found_target = False
    best_p_at_target = -1

    for threshold in thresholds:
        y_pred = (prop_probs_clean >= threshold).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true_clean, y_pred, average='binary', zero_division=0)

        #F2-Score (Weights recall 2x as much as precision)
        f2 = (5 * p * r) / (4 * p + r) if (4 * p + r) > 0 else 0

        #LOGIC: If we hit our 0.90 Recall goal, find the threshold that gives us the highest possible precision
        if r >= target_recall:
            if p > best_p_at_target:
                best_p_at_target = p
                best_metrics = {
                    "precision": p,
                    "recall": r,
                    "f1": f1,
                    "f2_score": f2,
                    "threshold": threshold
                }

        #If we never hit 0.90, keep the threshold with the highest recall
        elif best_p_at_target == -1:
            if r > best_metrics["recall"]:
                best_metrics = {
                    "precision": p, "recall": r, "f1": f1, "f2_score": f2, "threshold": threshold
                }

    return best_metrics

In [ ]:
#Initialize model - Load from local if exists, else from checkpoint
if (MODEL_DIR / "config.json").exists():
    print(f"Loading existing trained model from: {MODEL_DIR}")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
    model_already_trained = True
else:
    print(f"No existing model found. Initializing from: {"roberta-base"}")
    model = AutoModelForTokenClassification.from_pretrained("roberta-base", num_labels=3)
    model_already_trained = False

model.to(device)


In [ ]:
#Set up training arguments with optimized hyperparameters
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.5e-05,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.15,
    logging_steps=5,
    metric_for_best_model="loss",
    dataloader_pin_memory=False,
    disable_tqdm=True,
    report_to="none",
    load_best_model_at_end=True
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model loaded from disk. Skipping training.")

In [ ]:
#Evaluate performance on the test dataset
eval_trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=None,
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)
test_results = eval_trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F1 Score:  {test_results['eval_f1']:.4f}")
print(f"F2 Score:  {test_results['eval_f2_score']:.4f}")
print("="*30)

In [ ]:
#Compare results on train vs. test sets to ensure not overfitting
#Force evaluation on the Train set
train_results = trainer.evaluate(eval_dataset=tokenized_datasets["train"])
print(f"TRAIN Recall: {train_results['eval_recall']:.4f}")